# Meridian — Backbone Pre-training (Colab)

Runs Phase 1 TCP self-supervised pre-training on the hybrid RA-GDN + SWA backbone.
Checkpoints save to Google Drive every 1 000 steps; re-run all cells to auto-resume
after a session drop.

**Run `colab_fda.ipynb` first** to generate `FDA_SYNTH_*.f32bin` shards and push them
to `meridian-fda-synth` on HuggingFace — this notebook downloads and unpacks them
so the pretrain objective sees the full augmented dataset (spec §11.4c).

**Run this before `colab_portfolio_return.ipynb`.**

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'
FDA_TAR_REPO  = f'{HF_USERNAME}/meridian-fda-synth'  # packed FDA tars from colab_fda.ipynb

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/pretrain', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',           exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub>=0.26',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download, HfApi
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file ──────────────────────────────────
import os, shutil

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

# symbols.txt lives at repo root, not inside code/, so fetch it separately.
symbols_path = '/content/meridian/symbols.txt'
if not os.path.exists(symbols_path):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='symbols.txt')
    shutil.copy(path, symbols_path)
    print(f'symbols.txt downloaded → {symbols_path}')
else:
    print('symbols.txt already present.')

In [ ]:
# ── 4. Download real shards (cached to Drive across sessions) ─────────────────
# Drive FUSE is flaky on dirs with many files (sporadic EIO on listdir).
# All Drive directory operations go through a retry wrapper.
import os, shutil, time

def _drive_retry(fn, what, attempts=6, base_sleep=3):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except OSError as e:
            last = e
            print(f'[{what}] Drive EIO (attempt {i + 1}/{attempts}): {e}; sleeping {(i + 1) * base_sleep}s')
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(
        f'{what} failed after {attempts} retries. Drive FUSE is likely stuck. '
        f'Remount with: from google.colab import drive; drive.flush_and_unmount(); '
        f'drive.mount("/content/drive", force_remount=True); then re-run.'
    ) from last

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'

def _has_data():
    if not os.path.exists(train_dir):
        return False
    entries = _drive_retry(lambda: os.listdir(train_dir), 'list train_dir')
    return any(f.endswith('.f32bin') for f in entries)

if not _has_data():
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(
                os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]),
                ignore_errors=True,
            )
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy real shards to local SSD (parallel) ───────────────────────────────
# Drive FUSE handles concurrent reads well — a thread pool gets 8–16× speedup
# over single-threaded shutil.copytree. Each file copy retries on EIO.
import os, shutil, time, concurrent.futures

local_data  = '/content/data/mmaps'
drive_data  = f'{DRIVE_DIR}/data/mmaps'

def _copy_one(src, dst, attempts=5, base_sleep=1):
    for i in range(attempts):
        try:
            if os.path.exists(dst):
                return
            shutil.copy2(src, dst)
            return
        except OSError as e:
            last = e
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(f'copy failed {src} → {dst}: {last}')

def _parallel_copy_dir(src_dir, dst_dir, workers=32):
    os.makedirs(dst_dir, exist_ok=True)
    entries = sorted(os.listdir(src_dir))
    jobs    = [(os.path.join(src_dir, f), os.path.join(dst_dir, f)) for f in entries]
    t0, done = time.time(), 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_copy_one, s, d) for s, d in jobs]
        for fut in concurrent.futures.as_completed(futures):
            fut.result()
            done += 1
            if done % 50 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                print(f'    {done}/{len(jobs)} ({rate:.0f} files/s)')
    print(f'  {src_dir} → {dst_dir}: {len(jobs)} files in {time.time() - t0:.1f}s')

if not os.path.exists(f'{local_data}/train'):
    print('Copying shards from Drive to local SSD...')
    for split in ('train', 'validation', 'test'):
        src_split = os.path.join(drive_data, split)
        if os.path.isdir(src_split):
            _parallel_copy_dir(src_split, os.path.join(local_data, split))
    # Also copy SPY regime label caches if present.
    for npy in ('spy_regime_labels_train.npy', 'spy_regime_labels_val.npy'):
        src = os.path.join(drive_data, npy)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(local_data, npy))
    print('Done.')
else:
    print('Local shards already present.')

n_train = len([f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin')])
print(f'Train shards on SSD: {n_train}')

In [ ]:
# ── 6. Download + unpack FDA synth tars into local SSD ────────────────────────
# colab_fda.ipynb packs FDA_SYNTH_*.f32bin into ~100 MiB WebDataset tars and
# pushes them to FDA_TAR_REPO. We pull those tars here and extract the shards
# directly into the train/ directory so the pretrain ShardDataset sees them.
# FDA shards are detected by the FDA_SYNTH_ prefix and weighted by fda_loss_weight.
import os, tarfile, shutil
from huggingface_hub import list_repo_files, hf_hub_download

local_train = f'{local_data}/train'
existing_synth = [f for f in os.listdir(local_train) if f.startswith('FDA_SYNTH_')]

if existing_synth:
    print(f'FDA synth shards already present on SSD: {len(existing_synth)} — skipping download.')
else:
    try:
        tar_files = [
            f for f in list_repo_files(FDA_TAR_REPO, repo_type='dataset')
            if f.endswith('.tar')
        ]
    except Exception as e:
        tar_files = []
        print(f'Could not list {FDA_TAR_REPO}: {e}')
        print('Continuing without FDA synth shards (run colab_fda.ipynb first for full quality).')

    if tar_files:
        print(f'Downloading and unpacking {len(tar_files)} FDA tar(s) from {FDA_TAR_REPO}...')
        extracted = 0
        for tar_name in sorted(tar_files):
            local_tar = hf_hub_download(
                repo_id=FDA_TAR_REPO, filename=tar_name, repo_type='dataset'
            )
            with tarfile.open(local_tar, 'r') as tf:
                for member in tf:
                    if not member.isfile() or not member.name.endswith('.f32bin'):
                        continue
                    dst = os.path.join(local_train, os.path.basename(member.name))
                    if os.path.exists(dst):
                        continue
                    with tf.extractfile(member) as src_f, open(dst, 'wb') as dst_f:
                        shutil.copyfileobj(src_f, dst_f)
                    extracted += 1
            os.remove(local_tar)  # free SSD space after unpacking
        print(f'Extracted {extracted} FDA synth shards.')

real_shards  = [f for f in os.listdir(local_train) if f.endswith('.f32bin') and not f.startswith('FDA_SYNTH_')]
synth_shards = [f for f in os.listdir(local_train) if f.startswith('FDA_SYNTH_')]
print(f'Real shards: {len(real_shards)}  |  FDA synth shards: {len(synth_shards)}')
if not synth_shards:
    print('WARNING: no FDA synth shards — training real-data-only (run colab_fda.ipynb for full spec compliance).')

In [ ]:
# ── 7. Check for existing pretrain checkpoint (auto-resume) ───────────────────
# Prefers the latest backbone_step{N}.pt over backbone_pretrain.pt so training
# continues from the most recent periodic save, not just the last best-val save.
import os, re, glob, shutil

DRIVE_CKPT_DIR = f'{DRIVE_DIR}/checkpoints/pretrain'
drive_best     = f'{DRIVE_CKPT_DIR}/backbone_pretrain.pt'

def _latest_step_ckpt(ckpt_dir):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, 'backbone_step*.pt')):
        m = re.search(r'backbone_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

latest_step = _latest_step_ckpt(DRIVE_CKPT_DIR)

# Fall back to HF only when Drive has nothing at all.
if latest_step is None and not os.path.exists(drive_best):
    try:
        path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='pretrain/backbone_pretrain.pt')
        shutil.copy(path, drive_best)
        print(f'Checkpoint downloaded from HF → {drive_best}')
    except Exception:
        print('No existing checkpoint on HF — starting fresh.')

if latest_step is not None:
    resume_from = latest_step
    print(f'Will resume from latest step checkpoint: {resume_from}')
elif os.path.exists(drive_best):
    resume_from = drive_best
    print(f'Will resume from best-val checkpoint: {resume_from}')
else:
    resume_from = None
    print('Starting fresh.')

In [ ]:
# ── 8. Auto-detect GPU + set hyperparameters ──────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  ({vram_gb:.0f} GB VRAM)')

# TF32 — free ~10–30% on Ampere/Hopper with negligible numerical impact.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# VRAM budget for pretrain (N=1 per step, B=128 spec-mandated):
#
#   Full BPTT (K=0) keeps the entire T=64 recurrent state chain in the
#   backward graph — peaks at ~45–50 GB with B=128.
#
#   Truncated BPTT (K=32) detaches the GDN state every 32 steps, halving
#   the peak to ~22–24 GB.  The forward pass is IDENTICAL — only the gradient
#   signal for long-range (>32-day) patterns through the recurrent state is
#   approximated.  On daily data this is a negligible quality tradeoff.
#
#   --gradient-checkpointing adds ~33% extra forward compute (recomputes
#   activations during backward) but saves ~30% VRAM on top of BPTT.
#   Do NOT combine with --compile on the recurrent backbone (causes crashes).
#
# B=128 is kept across all tiers ≥22 GB because InfoNCE contrastive accuracy
# degrades sharply with fewer in-batch negatives. Only the 16 GB tier drops
# batch size as an unavoidable last resort.

if vram_gb >= 75:    # A100 80GB / H100 — full spec, no compromises
    BATCH_SIZE  = 128
    BPTT        = 0      # full T=64 BPTT
    GRAD_CKPT   = False
    USE_COMPILE = True
elif vram_gb >= 35:  # A100 40GB — BPTT=32 fits B=128 without grad_ckpt
    BATCH_SIZE  = 128
    BPTT        = 32
    GRAD_CKPT   = False  # not needed at 40 GB; grad_ckpt adds ~33% compute cost
    USE_COMPILE = True
elif vram_gb >= 22:  # L4 24 GB / RTX 3090 24 GB — B=128 preserved via BPTT=32
    BATCH_SIZE  = 128    # keep spec batch size; BPTT=32 makes this fit at 24 GB
    BPTT        = 32     # halves recurrent state-chain VRAM: O(T) → O(K=32)
    GRAD_CKPT   = True   # belt-and-suspenders for 24 GB headroom
    USE_COMPILE = False  # compile + grad_ckpt conflict on recurrent GDN backbone
else:                # T4 16 GB / V100 16 GB — unavoidable quality tradeoff
    BATCH_SIZE  = 64
    BPTT        = 32
    GRAD_CKPT   = True
    USE_COMPILE = False
    print('WARNING: 16 GB tier — batch size reduced to 64 (spec deviation).')
    print('         InfoNCE negative diversity will be lower than the B=128 target.')
    print('         Upgrade to an L4/A100 runtime for full quality.')

WORKERS = min(8, os.cpu_count() or 1)
print(f'batch={BATCH_SIZE}  bptt={BPTT}  grad_ckpt={GRAD_CKPT}  compile={USE_COMPILE}  workers={WORKERS}')

In [ ]:
# ── 9. Pre-train backbone (auto-resumes from Drive) ───────────────────────────
import os, sys, subprocess

cmd = [
    sys.executable, '-u',
    '-m', 'train.pretrain',
    '--data-dir',                   '/content/data/mmaps',
    '--symbols-file',               '/content/meridian/symbols.txt',
    '--steps',                      '20000',
    '--batch-size',                 str(BATCH_SIZE),
    '--seq-len',                    '64',
    '--lr',                         '3e-4',
    '--min-lr',                     '1e-5',
    '--warmup-steps',               '3000',
    '--grad-clip',                  '1.0',
    '--temperature',                '0.1',
    '--lambda-contrastive',         '0.7',
    '--lambda-prediction',          '0.3',
    '--fda-loss-weight',            '0.5',
    '--positive-offset',            '4',
    '--multi-asset-corr-threshold', '0.6',
    '--negative-min-separation',    '50',
    '--log-every',                  '50',
    '--eval-every',                 '500',
    '--eval-max-batches',           '50',
    '--save-every',                 '1000',
    '--checkpoint-dir',             DRIVE_CKPT_DIR,
    '--workers',                    str(WORKERS),
]
if resume_from:
    cmd += ['--resume-from', resume_from]
if BPTT > 0:
    cmd += ['--bptt-horizon', str(BPTT)]
if GRAD_CKPT:
    cmd.append('--gradient-checkpointing')
if USE_COMPILE:
    cmd.append('--compile')

env = os.environ.copy()
env['PYTHONPATH']       = '/content/meridian'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Pre-training failed with return code {process.returncode}')
print('Pre-training complete.')

In [ ]:
# ── 10. Upload best checkpoint to HuggingFace ─────────────────────────────────
import os
from huggingface_hub import HfApi

best_ckpt = f'{DRIVE_CKPT_DIR}/backbone_pretrain.pt'
if not os.path.exists(best_ckpt):
    print('No best checkpoint found — nothing to upload.')
else:
    api = HfApi()
    api.upload_file(
        path_or_fileobj=best_ckpt,
        path_in_repo='pretrain/backbone_pretrain.pt',
        repo_id=HF_MODEL_REPO,
        repo_type='model',
    )
    print(f'Uploaded → {HF_MODEL_REPO}/pretrain/backbone_pretrain.pt')